# Example: reproduce oscillator evaporative-cooling graphics

This notebook demonstrates how to reproduce the same **oscillator** graphics from the
original notebook using the modular code in `evap_cooling.py`.

It generates the same families of plots:

- Sample temperature vs number of atoms.
- Normalized $N_i/N_0$ vs normalized $T_i/T_0$.
- Temperature vs cut-off temperature $Q$.
- Number of atoms vs cut-off temperature $Q$.


In [ ]:
import math
import mpmath as mp
import matplotlib.pyplot as plt

from evap_cooling import HarmonicTrapModel


In [ ]:
# Initial conditions (same order of magnitude as original notebook)
N0 = 1e7
T0 = 5e-5
omega = 2 * mp.pi * 100

# Evaporation schedule
Qc0 = 5e-4
dqc = 1e-7
n_steps = 2000
q_values = [Qc0 - i * dqc for i in range(1, n_steps + 1)]

model = HarmonicTrapModel(omega=float(omega))


In [ ]:
def nf(Ni, Ti, Mui, qc):
    etac = qc / Ti
    g3t = -mp.polylog(3, -mp.exp(Mui / (model.c.kb * Ti)) * mp.erf(mp.sqrt(etac)))
    g3 = -mp.polylog(3, -mp.exp(Mui / (model.c.kb * Ti)))
    g52b = -mp.polylog(5 / 2, -mp.exp(Mui / (model.c.kb * Ti) - etac))
    return (g3t / g3 - (2 / mp.sqrt(mp.pi)) * mp.sqrt(etac) * (g52b / g3)) * Ni


def ef(Ni, Ti, Mui, Ei, qc):
    etac = qc / Ti
    g52b = -mp.polylog(5 / 2, -mp.exp(Mui / (model.c.kb * Ti) - etac))
    g4 = -mp.polylog(4, -mp.exp(Mui / (model.c.kb * Ti)))
    g4t = -mp.polylog(4, -mp.exp(Mui / (model.c.kb * Ti)) * mp.erf(mp.sqrt(etac)))
    g72b = -mp.polylog(7 / 2, -mp.exp(Mui / (model.c.kb * Ti) - etac))
    return (
        g4t / g4
        - (2 / mp.sqrt(mp.pi)) * mp.sqrt(etac) * (g72b / g4)
        - (2 / (3 * mp.sqrt(mp.pi))) * (etac ** (3 / 2)) * (g52b / g4)
    ) * Ei


def nb(Ni, Ti, Mui, qc):
    etac = qc / Ti
    g3t = mp.polylog(3, mp.exp(Mui / (model.c.kb * Ti)) * mp.erf(mp.sqrt(etac)))
    g3 = mp.polylog(3, mp.exp(Mui / (model.c.kb * Ti)))
    g52b = mp.polylog(5 / 2, mp.exp(Mui / (model.c.kb * Ti) - etac))
    return (g3t / g3 - (2 / mp.sqrt(mp.pi)) * mp.sqrt(etac) * (g52b / g3)) * Ni


def eb(Ni, Ti, Mui, Ei, qc):
    etac = qc / Ti
    g52b = mp.polylog(5 / 2, mp.exp(Mui / (model.c.kb * Ti) - etac))
    g4 = mp.polylog(4, mp.exp(Mui / (model.c.kb * Ti)))
    g4t = mp.polylog(4, mp.exp(Mui / (model.c.kb * Ti)) * mp.erf(mp.sqrt(etac)))
    g72b = mp.polylog(7 / 2, mp.exp(Mui / (model.c.kb * Ti) - etac))
    return (
        g4t / g4
        - (2 / mp.sqrt(mp.pi)) * mp.sqrt(etac) * (g72b / g4)
        - (2 / (3 * mp.sqrt(mp.pi))) * (etac ** (3 / 2)) * (g52b / g4)
    ) * Ei


def n_mb(Ni, Ti, qc):
    etac = qc / Ti
    return (mp.erf(mp.sqrt(etac)) - (2 / mp.sqrt(mp.pi)) * mp.sqrt(etac) * mp.exp(-etac)) * Ni


def t_mb(Ti, qc):
    etac = qc / Ti
    return (
        mp.erf(mp.sqrt(etac))
        - (2 / mp.sqrt(mp.pi)) * mp.sqrt(etac) * mp.exp(-etac)
        - (4 / (3 * mp.sqrt(mp.pi))) * (etac ** (3 / 2)) * mp.exp(-etac)
    ) * Ti


In [ ]:
def initialize_branch(boson):
    x = model.solve_chemical_potential(
        n_atoms=N0,
        temperature=T0,
        boson=boson,
        guess=-11.8,
        bracket=(-14, -1e-6) if boson else (-14, -6),
    )
    mu0 = x * model.c.kb * T0
    e0 = model.energy(N0, T0, mu0, boson)
    return {
        "N": [float(N0)],
        "T": [float(T0)],
        "Mu": [float(mu0)],
        "E": [float(e0)],
        "Q": q_values,
        "Nf": [1.0],
        "Tf": [1.0],
    }


def initialize_mb():
    return {
        "N": [float(N0)],
        "T": [float(T0)],
        "Q": q_values,
        "Nf": [1.0],
        "Tf": [1.0],
    }


In [ ]:
evap_b = initialize_branch(boson=True)
evap_f = initialize_branch(boson=False)
evap_mb = initialize_mb()

for qc in q_values:
    # Bosons
    Ni, Ti, Mui, Ei = evap_b["N"][-1], evap_b["T"][-1], evap_b["Mu"][-1], evap_b["E"][-1]
    N1 = float(nb(Ni, Ti, Mui, qc))
    E1 = float(eb(Ni, Ti, Mui, Ei, qc))
    state_b = model.solve_state(N1, E1, boson=True, t_guess=Ti, mu_guess=Mui)
    evap_b["N"].append(state_b.n_atoms)
    evap_b["T"].append(state_b.temperature)
    evap_b["Mu"].append(state_b.chemical_potential)
    evap_b["E"].append(state_b.energy)
    evap_b["Nf"].append(state_b.n_atoms / N0)
    evap_b["Tf"].append(state_b.temperature / T0)

    # Fermions
    Ni, Ti, Mui, Ei = evap_f["N"][-1], evap_f["T"][-1], evap_f["Mu"][-1], evap_f["E"][-1]
    N1 = float(nf(Ni, Ti, Mui, qc))
    E1 = float(ef(Ni, Ti, Mui, Ei, qc))
    state_f = model.solve_state(N1, E1, boson=False, t_guess=Ti, mu_guess=Mui)
    evap_f["N"].append(state_f.n_atoms)
    evap_f["T"].append(state_f.temperature)
    evap_f["Mu"].append(state_f.chemical_potential)
    evap_f["E"].append(state_f.energy)
    evap_f["Nf"].append(state_f.n_atoms / N0)
    evap_f["Tf"].append(state_f.temperature / T0)

    # Maxwell-Boltzmann
    Ni, Ti = evap_mb["N"][-1], evap_mb["T"][-1]
    N1 = float(n_mb(Ni, Ti, qc))
    T1 = float(t_mb(Ti, qc))
    evap_mb["N"].append(N1)
    evap_mb["T"].append(T1)
    evap_mb["Nf"].append(N1 / N0)
    evap_mb["Tf"].append(T1 / T0)


In [ ]:
plt.figure(figsize=(12, 6))
plt.scatter(evap_b["T"], evap_b["N"], s=7, label="Bosons")
plt.scatter(evap_f["T"], evap_f["N"], s=7, label="Fermions")
plt.scatter(evap_mb["T"], evap_mb["N"], s=7, label="Maxwell-Boltzmann")
plt.xlabel("Sample temperature [K]")
plt.ylabel("Number of atoms")
plt.xscale("log")
plt.yscale("log")
plt.title("Oscillator: temperature vs number of atoms")
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(12, 6))
plt.scatter(evap_b["Tf"], evap_b["Nf"], s=7, label="Bosons")
plt.scatter(evap_f["Tf"], evap_f["Nf"], s=7, label="Fermions")
plt.scatter(evap_mb["Tf"], evap_mb["Nf"], s=7, label="Maxwell-Boltzmann")
plt.xlabel("$T_i / T_0$")
plt.ylabel("$N_i / N_0$")
plt.xscale("log")
plt.yscale("log")
plt.title("Oscillator: normalized number vs normalized temperature")
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(12, 6))
plt.scatter(evap_b["Q"], evap_b["Tf"][1:], s=7, label="Bosons")
plt.scatter(evap_f["Q"], evap_f["Tf"][1:], s=7, label="Fermions")
plt.scatter(evap_mb["Q"], evap_mb["Tf"][1:], s=7, label="Maxwell-Boltzmann")
plt.xlabel("Cut-off temperature Q [K]")
plt.ylabel("Normalized temperature $T_i/T_0$")
plt.xscale("log")
plt.yscale("log")
plt.title("Oscillator: normalized temperature vs cut-off")
plt.legend()
plt.show()


In [ ]:
plt.figure(figsize=(12, 6))
plt.scatter(evap_b["Q"], evap_b["Nf"][1:], s=7, label="Bosons")
plt.scatter(evap_f["Q"], evap_f["Nf"][1:], s=7, label="Fermions")
plt.scatter(evap_mb["Q"], evap_mb["Nf"][1:], s=7, label="Maxwell-Boltzmann")
plt.xlabel("Cut-off temperature Q [K]")
plt.ylabel("Normalized number $N_i/N_0$")
plt.xscale("log")
plt.yscale("log")
plt.title("Oscillator: normalized number vs cut-off")
plt.legend()
plt.show()
